<a href="https://colab.research.google.com/github/Samarth-27/Celebal-CEI/blob/main/week8_SamarthJain_Jecrc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Keyword and calculator Pipeline

Small single-agent setup that routes a query to one of three places based on what's in the text: a calculator, a keyword extractor, or a plain fallback response when nothing matches.

Main things this needs to cover: keyword-based routing, calling the right tool, and not crashing when the input is malformed (bad math expression, empty string, etc).


In [ ]:
# Calculator

def calculator(expression: str) -> str:
    """Evaluates math expression and returns a string."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [ ]:
# Keyword Extractor

def extract_keywords(text: str) -> list:
    """Rough keyword extraction grabs words longer than 4 characters."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

## Agent Logic

Routing is just keyword matching on the query:
- "calculate" in the text -> calculator
- "keyword" in the text -> keyword extractor
- anything else -> general fallback response


In [ ]:
# Agent Function

import re

def agent(query: str):
    query_lower = query.lower()

    try:
        if "calculate" in query_lower:
            # strip "calculate" out, otherwise eval() chokes on the extra word
            expression = query_lower.replace("calculate", "").strip()
            result = calculator(expression)
            if result == "Error in calculation":
                print(f"[log] calculation failed for: '{expression}'")
                return {"type": "error", "result": result}
            print(f"[log] calculator -> {expression}")
            return {"type": "calculation", "result": result}

        elif "keyword" in query_lower:
            # "extract"/"keywords" sneak into the result if I don't strip them first
            text_for_extraction = re.sub(r"\b(extract|keywords?|from)\b", "", query, flags=re.IGNORECASE)
            text_for_extraction = " ".join(text_for_extraction.split())
            result = extract_keywords(text_for_extraction)
            print(f"[log] keyword extractor -> found {len(result)}")
            return {"type": "keywords", "result": result}

        else:
            print("[log] nothing matched, going general")
            return {"type": "general", "result": f"General query: '{query}'"}

    except Exception as e:
        print(f"[log] something broke: {e}")
        return {"type": "error", "result": str(e)}

## Output Format

Every response comes back in this shape:

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```


In [ ]:
# quick check with a few sample queries

queries = [
    "find keywords Deep learning models require large datasets and heavy computation",
    "Calculate 156 * 3 - 20",
    "Who won the last cricket world cup?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: find keywords Deep learning models require large datasets and heavy computation
[log] keyword extractor -> found 5
Response: {'type': 'keywords', 'result': ['learning', 'datasets', 'models', 'large', 'computation']}
--------------------------------------------------
Query: Calculate 156 * 3 - 20
[log] calculator -> 156 * 3 - 20
Response: {'type': 'calculation', 'result': '448'}
--------------------------------------------------
Query: Who won the last cricket world cup?
[log] nothing matched, going general
Response: {'type': 'general', 'result': "General query: 'Who won the last cricket world cup?'"}
--------------------------------------------------


In [ ]:
# manual testing loop, type exit to quit

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

Enter query (type 'exit' to stop): Calculate 999 - 233
[log] calculator -> 999 - 233
Response: {'type': 'calculation', 'result': '766'}
Enter query (type 'exit' to stop): Find the keywords in Climate change is affecting global weather patterns significantly
[log] keyword extractor -> found 5
Response: {'type': 'keywords', 'result': ['weather', 'patterns', 'climate', 'global', 'change']}
Enter query (type 'exit' to stop): exit
